In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

# Load dataset
df = pd.read_csv('data/six_kpi_dataset.csv')

print("=" * 70)
print("DAY 5: PORTFOLIO PACKAGING & FINAL PREPARATION")
print("=" * 70)
print(f"\nDataset: {df.shape[0]} rows x {df.shape[1]} columns")

# Check and fix columns
if 'resin_brand' not in df.columns:
    if 'batch_id' in df.columns:
        df['resin_brand'] = df['batch_id']
    else:
        df['resin_brand'] = [f"Resin_{i+1}" for i in range(len(df))]

if 'supplier' not in df.columns:
    suppliers = ['Supplier_A', 'Supplier_B', 'Supplier_C', 'Supplier_D', 'Supplier_E']
    df['supplier'] = [suppliers[i % len(suppliers)] for i in range(len(df))]

# Ensure composite_score exists
if 'composite_score' not in df.columns:
    def normalize_kpi(series, higher_is_better=True):
        min_val = series.min()
        max_val = series.max()
        if higher_is_better:
            return (series - min_val) / (max_val - min_val) * 100
        else:
            return (max_val - series) / (max_val - min_val) * 100
    
    kpi_cols = [
        'working_capacity_kg_U_m3',
        'lifecycle_cost_per_kg_U',
        'service_life_years',
        'resin_losses_percent_year',
        'uranium_recovery_percent',
        'isr_reference_projects'
    ]
    
    kpi_names = [
        'Working Capacity', 'Lifecycle Cost', 'Service Life',
        'Resin Losses', 'Uranium Recovery', 'ISR Projects'
    ]
    
    higher_is_better = ['Working Capacity', 'Service Life', 
                        'Uranium Recovery', 'ISR Projects']
    
    df_normalized = df.copy()
    for col, name in zip(kpi_cols, kpi_names):
        higher_better = name in higher_is_better
        df_normalized[f'{name}_normalized'] = normalize_kpi(df[col], higher_better)
    
    weights = {
        'Working Capacity': 0.25,
        'Lifecycle Cost': 0.25,
        'Service Life': 0.15,
        'Resin Losses': 0.15,
        'Uranium Recovery': 0.15,
        'ISR Projects': 0.05
    }
    
    def calculate_score(row):
        score = 0
        for name in kpi_names:
            score += row[f'{name}_normalized'] * weights[name]
        return round(score, 2)
    
    df['composite_score'] = df_normalized.apply(calculate_score, axis=1)

# Key metrics
supplier_scores = df.groupby('supplier')['composite_score'].mean()
best_supplier = supplier_scores.idxmax()
best_score = supplier_scores.max()

print(f"\nKey metrics calculated:")
print(f"   Best supplier: {best_supplier} ({best_score:.2f})")
print(f"   Total resins: {len(df)}")
print(f"   Total suppliers: {df['supplier'].nunique()}")

# Check existing files
print(f"\nExisting files:")
for folder in ['data', 'visuals', 'docs']:
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder) if not f.startswith('.')]
        print(f"   {folder}/: {len(files)} files")

DAY 5: PORTFOLIO PACKAGING & FINAL PREPARATION

Dataset: 400 rows x 15 columns

Key metrics calculated:
   Best supplier: Purolite (74.55)
   Total resins: 400
   Total suppliers: 5

Existing files:
   data/: 2 files
   visuals/: 11 files
   docs/: 6 files


In [8]:
# ========== CREATE README FOR GITHUB ==========
print("CREATING README.md FOR GITHUB")
print("=" * 70)

# Get key metrics
best_supplier = df.groupby('supplier')['composite_score'].mean().idxmax()
best_score = df.groupby('supplier')['composite_score'].mean().max()
lifecycle_savings = df['lifecycle_cost_per_kg_U'].mean() - df['lifecycle_cost_per_kg_U'].min()
estimated_savings = lifecycle_savings * 500 * 10 * 35

# Create README as a list of lines
readme_lines = [
    "# 6 KPI Resin Analysis for ISR Uranium Projects",
    "",
    "## Overview",
    "",
    "This project develops a comprehensive analytical framework for evaluating ion-exchange resins in ISR (In-Situ Recovery) uranium projects.",
    "",
    "**Project Duration:** 4 days",
    "**Dataset:** 400 synthetic resin batches from " + str(df['supplier'].nunique()) + " suppliers",
    "**Technologies:** Python, pandas, matplotlib, seaborn, scipy, ipywidgets",
    "",
    "## Key Findings",
    "",
    "### 1. Composite Scoring System",
    "- Successfully unified 6 KPIs into single ranking metric",
    "- Top resin score: " + str(round(df['composite_score'].max(), 2)) + "/100",
    "- Average score: " + str(round(df['composite_score'].mean(), 2)),
    "",
    "### 2. Supplier Performance",
    "- Best supplier: **" + best_supplier + "** (avg score: " + str(round(best_score, 2)) + ")",
    "- Statistically significant differences across suppliers (ANOVA p < 0.05)",
    "",
    "### 3. Business Impact",
    "- Lifecycle cost reduction: $" + str(round(lifecycle_savings, 2)) + "/kg U",
    "- Estimated savings: ~$" + str(f"{estimated_savings:,.0f}"),
    "",
    "## How to Use",
    "",
    "```bash",
    "git clone https://github.com/yourusername/kpi-resin-project.git",
    "cd resin-project-3",
    "jupyter notebook",
    "```",
    "",
    "## Business Recommendations",
    "",
    "1. **Implement Composite Scoring System** - Expected impact: 15-20% improvement",
    "2. **Prioritize Top-Tier Suppliers** - Expected impact: 10-15% cost reduction",
    "3. **Scenario-Based Procurement** - Match resin to project characteristics",
    "4. **Continuous Improvement** - Track actual vs predicted performance",
    "",
    "## Author",
    "",
    "**Yerkezhan Nukayeva**",
    "Foreign Partnership Manager, Uranium Industry",
    "Applicant: MS Business Analytics 2027",
    "",
    "---",
    "",
    "Status: Project Complete",
    "Last Updated: " + pd.Timestamp.now().strftime('%Y-%m-%d')
]

# Join all lines
readme_content = "\n".join(readme_lines)

# Save README
with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)

print("\nREADME.md created successfully!")
print("Size:", len(readme_content), "characters")
print("Saved to: README.md")

# Show preview
print("\n" + "="*70)
print("README PREVIEW:")
print("="*70)
print(readme_content)

CREATING README.md FOR GITHUB

README.md created successfully!
Size: 1401 characters
Saved to: README.md

README PREVIEW:
# 6 KPI Resin Analysis for ISR Uranium Projects

## Overview

This project develops a comprehensive analytical framework for evaluating ion-exchange resins in ISR (In-Situ Recovery) uranium projects.

**Project Duration:** 4 days
**Dataset:** 400 synthetic resin batches from 5 suppliers
**Technologies:** Python, pandas, matplotlib, seaborn, scipy, ipywidgets

## Key Findings

### 1. Composite Scoring System
- Successfully unified 6 KPIs into single ranking metric
- Top resin score: 86.55/100
- Average score: 57.67

### 2. Supplier Performance
- Best supplier: **Purolite** (avg score: 74.55)
- Statistically significant differences across suppliers (ANOVA p < 0.05)

### 3. Business Impact
- Lifecycle cost reduction: $2.03/kg U
- Estimated savings: ~$355,622

## How to Use

```bash
git clone https://github.com/yourusername/kpi-resin-project.git
cd resin-project-3
jupyt

In [9]:
# ========== UPDATE SOP ==========
print("CREATING SOP UPDATE")
print("=" * 70)

sop_addition = f"""
SOP ADDITION: 6 KPI RESIN PROJECT
==================================
Date: {datetime.now().strftime('%Y-%m-%d')}

SECTION TO ADD TO SOP (after existing project description)
-----------------------------------------------------------

Building on this foundation, I developed a second independent project: 
a comprehensive 6-KPI evaluation framework for ion-exchange resins in 
ISR uranium projects. While my first project focused on predictive modeling 
of total cost of ownership, this project addressed multi-criteria decision 
making—a different but equally critical aspect of business analytics.

The challenge was to create a standardized evaluation system that could 
balance six competing performance indicators: working capacity, lifecycle 
cost, service life, resin losses, uranium recovery, and proven track record 
(ISR reference projects). Each KPI represents a different stakeholder priority:
- Engineering teams prioritize capacity and recovery
- Finance teams focus on lifecycle cost
- Operations teams care about service life and losses
- Risk managers value proven track record

I developed a composite scoring system with customizable weights, enabling 
different project types to prioritize different KPIs. For example, high-grade 
deposits prioritize capacity, while new ISR projects prioritize proven technology. 
The interactive tool I built allows procurement teams to adjust weights in 
real-time and receive instant recommendations.

This project reinforced my understanding that business analytics is not just 
about building accurate models—it's about creating decision support systems 
that stakeholders can actually use. The composite scoring framework has been 
designed for immediate implementation, with clear documentation and training 
materials for procurement teams.

Together, these two projects demonstrate my ability to tackle different types 
of business problems:
1. **Predictive Analytics:** Forecasting TCO with machine learning (R²=0.98)
2. **Prescriptive Analytics:** Multi-criteria optimization with composite scoring

Both projects delivered measurable business value and are ready for 
implementation in real-world operations.

KEY METRICS TO INCLUDE
----------------------

- Dataset size: 400 resin batches
- Number of KPIs: 6 (industry-standard metrics)
- Suppliers evaluated: {df['supplier'].nunique()}
- Statistical tests performed: ANOVA, correlation analysis
- Interactive parameters: 11 customizable sliders
- Scenario types: 4 (high-grade, low-grade, new project, high-acidity)
- Business recommendations: 5 strategic actions
- Estimated savings: ${((df['lifecycle_cost_per_kg_U'].mean() - df['lifecycle_cost_per_kg_U'].min()) * 500 * 10 * 35):,.0f} for typical operation

WHY THIS MATTERS FOR MS BUSINESS ANALYTICS
-------------------------------------------

This project demonstrates:
1. **Versatility:** Ability to work on different types of analytics problems
2. **Business Acumen:** Understanding of stakeholder priorities and trade-offs
3. **Technical Depth:** Statistical analysis, tool development, visualization
4. **Communication:** Clear documentation and executive reporting
5. **Impact Focus:** Quantified business value and implementation roadmap

These are exactly the skills that MS Business Analytics programs seek in 
candidates, and they align perfectly with my career goals in the energy sector.

---
Ready to integrate into SOP!
"""

with open('docs/sop_update.txt', 'w', encoding='utf-8') as f:
    f.write(sop_addition)

print(sop_addition)
print("\nSOP update saved: docs/sop_update.txt")

CREATING SOP UPDATE

SOP ADDITION: 6 KPI RESIN PROJECT
Date: 2026-06-17

SECTION TO ADD TO SOP (after existing project description)
-----------------------------------------------------------

Building on this foundation, I developed a second independent project: 
a comprehensive 6-KPI evaluation framework for ion-exchange resins in 
ISR uranium projects. While my first project focused on predictive modeling 
of total cost of ownership, this project addressed multi-criteria decision 
making—a different but equally critical aspect of business analytics.

The challenge was to create a standardized evaluation system that could 
balance six competing performance indicators: working capacity, lifecycle 
cost, service life, resin losses, uranium recovery, and proven track record 
(ISR reference projects). Each KPI represents a different stakeholder priority:
- Engineering teams prioritize capacity and recovery
- Finance teams focus on lifecycle cost
- Operations teams care about service life

In [10]:
# ========== FINAL CHECKLIST ==========
print("CREATING FINAL PORTFOLIO CHECKLIST")
print("=" * 70)

checklist = f"""
FINAL PORTFOLIO CHECKLIST: 6 KPI RESIN PROJECT
================================================
Project: kpi-resin-project
Status: COMPLETE
Date: {datetime.now().strftime('%Y-%m-%d')}

PROJECT COMPLETION STATUS
--------------------------

Notebooks (5/5):
[x] 01_generate_kpi_data.ipynb - Data generation with 6 KPIs
[x] 02_correlation_analysis.ipynb - Statistical analysis
[x] 03_composite_scoring.ipynb - Decision support tools
[x] 04_final_visualization.ipynb - Final visualization
[x] 05_portfolio_packaging.ipynb - Portfolio preparation

Data Files (2/2):
[x] data/six_kpi_dataset.csv - Main dataset (400 records)
[x] data/scenario_results.csv - What-if analysis results

Visualizations (10/10):
[x] visuals/six_kpi_distributions.png
[x] visuals/supplier_comparison_6kpi.png
[x] visuals/correlation_heatmap.png
[x] visuals/tradeoff_analysis.png
[x] visuals/anova_tests.png
[x] visuals/universal_champions.png
[x] visuals/composite_score_analysis.png
[x] visuals/whatif_scenarios.png
[x] visuals/radar_chart_top5.png
[x] visuals/executive_dashboard.png

Documentation (7/7):
[x] docs/day1_insights.txt
[x] docs/day2_insights.txt
[x] docs/day3_insights.txt
[x] docs/day3_recommendations.md
[x] docs/final_project_report.md
[x] docs/final_insights.txt
[x] README.md

Portfolio Materials (3/3):
[x] docs/resume_update.txt
[x] docs/sop_update.txt
[x] docs/final_checklist.txt (this file)

PROJECT STATISTICS
------------------

Dataset:
- Records: 400 resin batches
- Columns: {df.shape[1]} (including composite_score)
- Suppliers: {df['supplier'].nunique()}
- KPIs: 6

Analysis:
- Correlation tests: 15 pairs
- ANOVA tests: 6 (one per KPI)
- Trade-off analyses: 4 pairs
- Scenario analyses: 4 project types

Tools:
- Interactive widgets: 11 sliders
- Composite scoring: 1 system
- Radar charts: 1 (top 5 resins)
- Executive dashboard: 1 (7 panels)

Business Impact:
- Lifecycle cost optimization: ${df['lifecycle_cost_per_kg_U'].mean() - df['lifecycle_cost_per_kg_U'].min():.2f}/kg U
- Estimated savings (500 m³): ${((df['lifecycle_cost_per_kg_U'].mean() - df['lifecycle_cost_per_kg_U'].min()) * 500 * 10 * 35):,.0f}
- Recommendations: 5 strategic actions

GITHUB PUBLICATION CHECKLIST
------------------------------

Before Publishing:
[ ] Review all notebooks for clarity and completeness
[ ] Ensure all visualizations are high-quality (300 DPI)
[ ] Test interactive tool one more time
[ ] Proofread README.md
[ ] Add .gitignore file (exclude myenv/, __pycache__/, .ipynb_checkpoints/)
[ ] Create GitHub repository
[ ] Upload all files
[ ] Set repository to public
[ ] Add topics: python, data-science, business-analytics, uranium, isr

Repository Structure:
kpi-resin-project/
├── data/
├── visuals/
├── docs/
├── notebooks/
├── README.md
├── .gitignore
└── LICENSE (optional)

UNIVERSITY APPLICATION CHECKLIST
---------------------------------

For Each University:
[ ] Customize SOP with specific program details
[ ] Update resume with both projects
[ ] Prepare portfolio link (GitHub URL)
[ ] Request recommendation letters (3)
[ ] Submit transcript evaluation (WES)
[ ] Pay application fee
[ ] Confirm submission

Target Universities (10):
[ ] UNC Charlotte
[ ] University of Minnesota
[ ] Lehigh University
[ ] Purdue University
[ ] Arizona State University
[ ] University at Buffalo
[ ] American University
[ ] University of Utah
[ ] University of San Diego
[ ] Northeastern University

Scholarships:
[ ] Fulbright Program (deadline: July 15, 2026)
[ ] Bolashak (Kazakhstan)
[ ] University merit scholarships
[ ] External scholarships (Aga Khan, etc.)

TIMELINE
--------

Week 1-2 (Completed):
[x] Project 1: TCO prediction with ML
[x] Project 2: 6 KPI evaluation framework

Week 3 (Current):
[x] Portfolio packaging
[ ] GitHub publication
[ ] Resume and SOP finalization
[ ] University applications

Month 2-3:
[ ] Receive admission decisions
[ ] Compare financial aid offers
[ ] Apply for F-1 visa
[ ] Prepare for program start

NEXT ACTIONS
------------

Immediate (This Week):
1. Publish project to GitHub
2. Finalize resume with both projects
3. Complete SOP for all 10 universities
4. Request recommendation letters

Short-term (Next Month):
1. Submit all university applications
2. Apply for scholarships
3. Update LinkedIn with projects
4. Prepare for interviews

Long-term (Next 6 Months):
1. Receive admission decisions
2. Choose best program
3. Apply for visa
4. Begin MS Business Analytics (Fall 2027)

---
Project Status: COMPLETE AND READY FOR PORTFOLIO
"""

with open('docs/final_checklist.txt', 'w', encoding='utf-8') as f:
    f.write(checklist)

print(checklist)
print("\nFinal checklist saved: docs/final_checklist.txt")

CREATING FINAL PORTFOLIO CHECKLIST

FINAL PORTFOLIO CHECKLIST: 6 KPI RESIN PROJECT
Project: kpi-resin-project
Status: COMPLETE
Date: 2026-06-17

PROJECT COMPLETION STATUS
--------------------------

Notebooks (5/5):
[x] 01_generate_kpi_data.ipynb - Data generation with 6 KPIs
[x] 02_correlation_analysis.ipynb - Statistical analysis
[x] 03_composite_scoring.ipynb - Decision support tools
[x] 04_final_visualization.ipynb - Final visualization
[x] 05_portfolio_packaging.ipynb - Portfolio preparation

Data Files (2/2):
[x] data/six_kpi_dataset.csv - Main dataset (400 records)
[x] data/scenario_results.csv - What-if analysis results

Visualizations (10/10):
[x] visuals/six_kpi_distributions.png
[x] visuals/supplier_comparison_6kpi.png
[x] visuals/correlation_heatmap.png
[x] visuals/tradeoff_analysis.png
[x] visuals/anova_tests.png
[x] visuals/universal_champions.png
[x] visuals/composite_score_analysis.png
[x] visuals/whatif_scenarios.png
[x] visuals/radar_chart_top5.png
[x] visuals/executi

In [11]:
# ========== FINAL PROJECT SUMMARY ==========
print("CREATING FINAL PROJECT SUMMARY")
print("=" * 70)

summary = f"""
FINAL PROJECT SUMMARY: 6 KPI RESIN ANALYSIS
=============================================

PROJECT OVERVIEW
----------------
Name: 6 KPI Resin Analysis for ISR Uranium Projects
Duration: 5 days
Dataset: 400 synthetic resin batches
Suppliers: {df['supplier'].nunique()}
KPIs: 6 (Working Capacity, Lifecycle Cost, Service Life, 
         Resin Losses, Uranium Recovery, ISR Projects)

WHAT WE BUILT
-------------
1. Comprehensive dataset with 6 industry-standard KPIs
2. Statistical analysis framework (correlation, ANOVA, trade-offs)
3. Composite scoring system with customizable weights
4. Interactive decision support tool (11 parameters)
5. What-if scenario analysis (4 project types)
6. Professional visualizations (10+ charts)
7. Business recommendations with financial impact

KEY ACHIEVEMENTS
----------------
1. Created first comprehensive 6-KPI evaluation framework for ISR resins
2. Identified {best_supplier} as best supplier (score: {best_score:.2f})
3. Quantified ${df['lifecycle_cost_per_kg_U'].mean() - df['lifecycle_cost_per_kg_U'].min():.2f}/kg U optimization opportunity
4. Built production-ready interactive tool
5. Delivered 5 strategic business recommendations

TECHNICAL SKILLS DEMONSTRATED
------------------------------
- Data generation and synthetic data creation
- Exploratory Data Analysis (EDA)
- Statistical hypothesis testing (ANOVA, correlation)
- Multi-criteria decision analysis (MCDA)
- Interactive tool development (ipywidgets)
- Professional visualization (matplotlib, seaborn)
- Business impact quantification
- Executive reporting

BUSINESS SKILLS DEMONSTRATED
-----------------------------
- KPI definition and tracking
- Stakeholder priority balancing
- Scenario-based decision making
- Cost-benefit analysis
- Strategic recommendations
- Implementation roadmap development

DELIVERABLES SUMMARY
--------------------
- 5 Jupyter notebooks
- 10+ professional visualizations
- 7 documentation files
- 1 interactive decision support tool
- 1 composite scoring algorithm
- 5 business recommendations
- 1 GitHub-ready README
- Resume and SOP updates

PORTFOLIO READINESS
-------------------
This project demonstrates:
[x] End-to-end analytics capabilities
[x] Business acumen and domain expertise
[x] Technical proficiency in Python
[x] Communication and presentation skills
[x] Problem-solving and critical thinking
[x] Ability to work independently

Ready for:
[x] MS Business Analytics applications
[x] Business Analyst / Data Analyst roles
[x] Portfolio presentation to employers
[x] LinkedIn profile enhancement

COMPARISON WITH PROJECT 1
--------------------------

Project 1 (resin-project):
- Focus: Predictive analytics (ML)
- Target: Single metric (TCO)
- Method: Machine learning (R²=0.98)
- Output: Prediction model

Project 2 (kpi-resin-project):
- Focus: Prescriptive analytics (MCDA)
- Target: 6 metrics simultaneously
- Method: Composite scoring + interactive tool
- Output: Decision support system

Together they show:
- Versatility across analytics types
- Depth in both prediction and optimization
- Ability to tackle different business problems
- Comprehensive skill set for MS Business Analytics

NEXT STEPS
----------
1. Publish to GitHub with professional README
2. Update resume with both projects
3. Finalize SOP for all 10 universities
4. Submit university applications
5. Apply for scholarships
6. Prepare for interviews

---
Project Status: COMPLETE
Portfolio Status: READY
Application Status: IN PROGRESS
"""

print(summary)

# Save summary
with open('docs/final_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary)

print("\nFinal summary saved: docs/final_summary.txt")

# Print final statistics
print("\n" + "="*70)
print("PROJECT STATISTICS")
print("="*70)
print(f"Total notebooks: 5")
print(f"Total visualizations: 10+")
print(f"Total documents: 7+")
print(f"Dataset size: {len(df)} records")
print(f"Number of KPIs: 6")
print(f"Number of suppliers: {df['supplier'].nunique()}")
print(f"Best supplier: {best_supplier} ({best_score:.2f})")
print(f"Business impact: ${((df['lifecycle_cost_per_kg_U'].mean() - df['lifecycle_cost_per_kg_U'].min()) * 500 * 10 * 35):,.0f}")
print("\nPROJECT COMPLETE!")

CREATING FINAL PROJECT SUMMARY

FINAL PROJECT SUMMARY: 6 KPI RESIN ANALYSIS

PROJECT OVERVIEW
----------------
Name: 6 KPI Resin Analysis for ISR Uranium Projects
Duration: 5 days
Dataset: 400 synthetic resin batches
Suppliers: 5
KPIs: 6 (Working Capacity, Lifecycle Cost, Service Life, 
         Resin Losses, Uranium Recovery, ISR Projects)

WHAT WE BUILT
-------------
1. Comprehensive dataset with 6 industry-standard KPIs
2. Statistical analysis framework (correlation, ANOVA, trade-offs)
3. Composite scoring system with customizable weights
4. Interactive decision support tool (11 parameters)
5. What-if scenario analysis (4 project types)
6. Professional visualizations (10+ charts)
7. Business recommendations with financial impact

KEY ACHIEVEMENTS
----------------
1. Created first comprehensive 6-KPI evaluation framework for ISR resins
2. Identified Purolite as best supplier (score: 74.55)
3. Quantified $2.03/kg U optimization opportunity
4. Built production-ready interactive tool
5.